[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](http://colab.research.google.com/github/masanorihirano/pams/blob/main/examples/black_scholes.ipynb)

In [ ]:
# Please remove comment-out if necessary
#! pip install pams matplotlib
# pams 0.2.2 or older does not have the agent-parallel runners; install pams from GitHub instead:
#! pip install git+https://github.com/masanorihirano/pams.git matplotlib

# Black-Scholes workload

This example corresponds to the `black_scholes` sample, a port of BlackScholes of plhamJ.
`WorkloadFCNAgent` mimics agents that run heavy computations, such as Monte Carlo simulations, before they decide their orders.
Every time it is asked to submit orders until the time `bsLoadShutdown`, it prices a European call option by a Black-Scholes Monte Carlo simulation of `bsNumSamples` sample paths with `bsNumSteps` time steps each.
Then, it submits orders in the same way as `FCNAgent` with the probability `orderRate`.
The estimated option prices are only summed up to `bs_sum` and are not used to decide the orders; the Monte Carlo simulation is just a CPU workload.

The sample is a benchmark of the runners.
The same simulation can be run by `SequentialRunner`, `MultiThreadAgentParallelRunner`, and `MultiProcessAgentParallelRunner`, and the results are identical.
The last section compares their execution times.

`MultiThreadAgentParallelRunner` and `MultiProcessAgentParallelRunner` are not included in pams 0.2.2 or older.
If the installed pams does not have them, install pams from GitHub, e.g., `pip install git+https://github.com/masanorihirano/pams.git`.

In [ ]:
import copy
import math
import multiprocessing
import os
import random
import time
from typing import Any
from typing import Dict
from typing import List
from typing import Tuple
from typing import Type
from typing import Union

import matplotlib.pyplot as plt
import numpy as np

from pams import Cancel
from pams import Market
from pams import Order
from pams.agents import FCNAgent
from pams.logs.market_step_loggers import MarketStepSaver
from pams.runners import MultiProcessAgentParallelRunner
from pams.runners import MultiThreadAgentParallelRunner
from pams.runners import SequentialRunner
from pams.utils import JsonRandom

In [ ]:
def normal_cdf(x: float) -> float:
    # the cumulative distribution function of the standard normal distribution
    return (1.0 + math.erf(x / math.sqrt(2.0))) / 2.0


class BlackScholes:
    def __init__(
        self,
        prng: random.Random,
        initial_price: float,
        strike_price: float,
        risk_free_rate: float,
        volatility: float,
        maturity_time: float,
    ) -> None:
        self.prng = prng
        self.initial_price = initial_price
        self.strike_price = strike_price
        self.risk_free_rate = risk_free_rate
        self.volatility = volatility
        self.maturity_time = maturity_time

    def compute(self, n_samples: int, n_steps: int) -> float:
        # the price of the underlying asset follows a geometric Brownian motion, which is simulated by the Euler method
        r = self.risk_free_rate
        sigma = self.volatility
        dt = self.maturity_time / n_steps
        sqrt_dt = math.sqrt(dt)
        total = 0.0
        for _ in range(n_samples):
            price = self.initial_price
            for _ in range(n_steps):
                price += price * r * dt + price * self.prng.gauss(mu=0.0, sigma=1.0) * sigma * sqrt_dt
            total += max(price - self.strike_price, 0.0)
        return math.exp(-r * self.maturity_time) * (total / n_samples)

    def theoretical_price(self) -> float:
        # the Black-Scholes formula
        sigma_sqrt_t = self.volatility * math.sqrt(self.maturity_time)
        d1 = (
            math.log(self.initial_price / self.strike_price)
            + (self.risk_free_rate + self.volatility**2 / 2) * self.maturity_time
        ) / sigma_sqrt_t
        d2 = d1 - sigma_sqrt_t
        return self.initial_price * normal_cdf(x=d1) - self.strike_price * math.exp(
            -self.risk_free_rate * self.maturity_time
        ) * normal_cdf(x=d2)

In [ ]:
class WorkloadFCNAgent(FCNAgent):
    def setup(
        self,
        settings: Dict[str, Any],
        accessible_markets_ids: List[int],
        *args: Any,
        **kwargs: Any,
    ) -> None:
        super().setup(settings=settings, accessible_markets_ids=accessible_markets_ids)
        self.bs_workload: bool = settings.get("bsWorkload", False)
        self.bs_load_shutdown: int = settings.get("bsLoadShutdown", -1)
        self.bs_n_samples: int = settings.get("bsNumSamples", 0)
        self.bs_n_steps: int = settings.get("bsNumSteps", 0)
        if self.bs_workload and (self.bs_n_samples < 1 or self.bs_n_steps < 1):
            raise ValueError("bsNumSamples and bsNumSteps have to be positive if bsWorkload is true")
        if "orderRate" in settings:
            json_random: JsonRandom = JsonRandom(prng=self.prng)
            self.order_rate: float = json_random.random(json_value=settings["orderRate"])
        else:
            self.order_rate = 0.1
        if not 0.0 <= self.order_rate <= 1.0:
            raise ValueError("orderRate have to be between 0.0 and 1.0")
        self.bs_sum: float = 0.0

    def is_workload_on(self, time: int) -> bool:
        # the workload is never shut down if bsLoadShutdown is 0 or negative
        return self.bs_workload and (self.bs_load_shutdown <= 0 or time < self.bs_load_shutdown)

    def submit_orders(self, markets: List[Market]) -> List[Union[Order, Cancel]]:
        accessible_markets = [market for market in markets if self.is_market_accessible(market_id=market.market_id)]
        if len(accessible_markets) == 0:
            return []
        if self.is_workload_on(time=accessible_markets[0].get_time()):
            black_scholes = BlackScholes(
                prng=self.prng,
                initial_price=100.0,
                strike_price=100.0,
                risk_free_rate=0.1,
                volatility=0.3,
                maturity_time=3.0,
            )
            self.bs_sum += black_scholes.compute(n_samples=self.bs_n_samples, n_steps=self.bs_n_steps)
        # randomly submit (or not) orders
        if self.prng.random() > self.order_rate:
            return []
        return super().submit_orders(markets=markets)

## Monte Carlo pricing

The workload prices a European call option with the initial price 100, the strike price 100, the risk-free rate 0.1, the volatility 0.3, and the maturity 3.
The Monte Carlo estimate approaches the price given by the Black-Scholes formula as the number of sample paths increases.
The Euler method with a finite number of time steps also adds a small bias.

In [ ]:
black_scholes = BlackScholes(
    prng=random.Random(42),
    initial_price=100.0,
    strike_price=100.0,
    risk_free_rate=0.1,
    volatility=0.3,
    maturity_time=3.0,
)
theoretical_price = black_scholes.theoretical_price()
print(f"theoretical price: {theoretical_price:.4f}")
n_samples_list = [10, 30, 100, 300, 1000, 3000]
estimates = []
for n_samples in n_samples_list:
    estimate = black_scholes.compute(n_samples=n_samples, n_steps=100)
    estimates.append(estimate)
    print(f"{n_samples} sample paths: {estimate:.4f}")

In [ ]:
plt.plot(n_samples_list, estimates, marker="o", label="Monte Carlo (100 time steps)")
plt.axhline(theoretical_price, color="black", linestyle="--", label="Black-Scholes formula")
plt.xscale("log")
plt.xlabel("number of sample paths")
plt.ylabel("option price")
plt.legend()
plt.show()

## Simulation

The config is the same as `samples/black_scholes/config.json`.
There are three spot markets and an index market of them.
Each spot market has 20 `WorkloadFCNAgent`s, the index market has 10 `WorkloadFCNAgent`s, and an `ArbitrageAgent` trades between the index market and the spot markets.
All the `WorkloadFCNAgent`s are asked to submit orders in each step because `maxNormalOrders` is very large.
Each `WorkloadFCNAgent` processes a small workload of 10 sample paths with 10 time steps until the time 50, and submits orders with the probability 0.1.
`FundamentalPriceShock` lowers the fundamental price of `SpotMarket-1` by 10% at the time 0.

In [ ]:
config = {
	"simulation": {
		"markets": ["SpotMarket-1", "SpotMarket-2", "SpotMarket-3", "IndexMarket-I"],
		"agents": ["FCNAgents-1", "FCNAgents-2", "FCNAgents-3", "FCNAgents-I", "ArbitrageAgents"],
		"sessions": [
			{	"sessionName": 1,
				"iterationSteps": 100,
				"withOrderPlacement": True,
				"withOrderExecution": True,
				"withPrint": True,
				"maxNormalOrders": 10000000000, "MEMO": "EVERYONE",
				"maxHighFrequencyOrders": 1,
				"events": ["FundamentalPriceShock"]
			}
		]
	},

	"FundamentalPriceShock": {
		"class": "FundamentalPriceShock",
		"target": "SpotMarket-1",
		"triggerTime": 0,           "MEMO": "At the beginning of the session",
		"priceChangeRate": -0.1,    "MEMO": "Sign: negative for down; positive for up; zero for no change",
		"enabled": True
	},

	"SpotMarket": {
		"class": "Market",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000
	},
	"SpotMarket-1": {
		"extends": "SpotMarket"
	},
	"SpotMarket-2": {
		"extends": "SpotMarket"
	},
	"SpotMarket-3": {
		"extends": "SpotMarket"
	},

	"IndexMarket-I": {
		"class": "IndexMarket",
		"tickSize": 0.00001,
		"marketPrice": 300.0,
		"outstandingShares": 25000,
		"markets": ["SpotMarket-1", "SpotMarket-2", "SpotMarket-3"]
	},

	"FCNAgent": {
		"class": "WorkloadFCNAgent",
		"numAgents": 20,

		"MEMO": "Agent class",
		"assetVolume": 50,
		"cashAmount": 10000,

		"MEMO": "FCNAgent class",
		"fundamentalWeight": {"expon": [1.0]},
		"chartWeight": {"expon": [0.0]},
		"noiseWeight": {"expon": [1.0]},
		"noiseScale": 0.001,
		"timeWindowSize": [100, 200],
		"orderMargin": [0.0, 0.1],

		"MEMO": "WorkloadFCNAgent class",
		"bsWorkload": True,
		"bsLoadShutdown": 50,     "MEMO": "The workload is shut down from the time 50",
		"bsNumSamples": 10,
		"bsNumSteps": 10,
		"orderRate": 0.1
	},
	"FCNAgents-1": {
		"extends": "FCNAgent",
		"markets": ["SpotMarket-1"]
	},
	"FCNAgents-2": {
		"extends": "FCNAgent",
		"markets": ["SpotMarket-2"]
	},
	"FCNAgents-3": {
		"extends": "FCNAgent",
		"markets": ["SpotMarket-3"]
	},
	"FCNAgents-I": {
		"extends": "FCNAgent",
		"numAgents": 10,
		"markets": ["IndexMarket-I"],
		"fundamentalWeight": {"expon": [0.5]}
	},

	"ArbitrageAgents": {
		"class": "ArbitrageAgent",
		"numAgents": 1,

		"markets": ["IndexMarket-I", "SpotMarket-1", "SpotMarket-2", "SpotMarket-3"],
		"assetVolume": 50,      "NOTE": "per market",
		"cashAmount": 150000,   "NOTE": "total",

		"orderVolume": 1,
		"orderThresholdPrice": 1.0
	}
}

In [ ]:
def run(
    config: Dict, runner_class: Type[SequentialRunner] = SequentialRunner, seed: int = 42
) -> Tuple[MarketStepSaver, SequentialRunner, float]:
    saver = MarketStepSaver()
    runner = runner_class(
        settings=copy.deepcopy(config),
        prng=random.Random(seed),
        logger=saver,
    )
    runner.class_register(cls=WorkloadFCNAgent)
    start = time.perf_counter()
    runner.main()
    elapsed_time = time.perf_counter() - start
    return saver, runner, elapsed_time

In [ ]:
saver, runner, elapsed_time = run(config=config)
print(f"elapsed time: {elapsed_time:.2f} s")

In [ ]:
market_names = ["SpotMarket-1", "SpotMarket-2", "SpotMarket-3", "IndexMarket-I"]
market_prices = {
    market_name: dict(sorted(map(lambda x: (x["market_time"], x["market_price"]), filter(lambda x: x["market_name"] == market_name, saver.market_step_logs))))
    for market_name in market_names
}
fundamental_prices = {
    market_name: dict(sorted(map(lambda x: (x["market_time"], x["fundamental_price"]), filter(lambda x: x["market_name"] == market_name, saver.market_step_logs))))
    for market_name in market_names
}

In [ ]:
for market_name in market_names:
    line, = plt.plot(list(market_prices[market_name].keys()), list(market_prices[market_name].values()), label=market_name)
    plt.plot(list(fundamental_prices[market_name].keys()), list(fundamental_prices[market_name].values()), color=line.get_color(), linestyle="--")
plt.xlabel("ticks")
plt.ylabel("market price (dashed: fundamental price)")
plt.legend()
plt.show()

The market price of `SpotMarket-1` falls toward its new fundamental price 270.
The fundamental price of `IndexMarket-I` is the average of the fundamental prices of the spot markets weighted by their outstanding shares, i.e., 290, and the market price of `IndexMarket-I` also falls toward it.

## Workload

Each `WorkloadFCNAgent` processes the workload once in each step until the time `bsLoadShutdown`, i.e., 50 times in this simulation.
An estimate by only 10 sample paths is rough, but the average of the 50 estimates of each agent is close to the price given by the Black-Scholes formula.
The small difference comes from the Monte Carlo error and the bias of the Euler method with 10 time steps.

In [ ]:
workload_agents = [agent for agent in runner.simulator.agents if isinstance(agent, WorkloadFCNAgent)]
n_iteration_steps = config["simulation"]["sessions"][0]["iterationSteps"]
mean_estimates = [
    agent.bs_sum / sum(agent.is_workload_on(time=t) for t in range(n_iteration_steps))
    for agent in workload_agents
]
print(f"number of WorkloadFCNAgents: {len(workload_agents)}")
print(f"average of the estimates: {np.mean(mean_estimates):.4f} (theoretical price: {theoretical_price:.4f})")
plt.hist(mean_estimates, bins=15)
plt.axvline(theoretical_price, color="black", linestyle="--", label="Black-Scholes formula")
plt.xlabel("average of the estimates of each agent")
plt.ylabel("number of agents")
plt.legend()
plt.show()

## Runners

`MultiThreadAgentParallelRunner` and `MultiProcessAgentParallelRunner` parallelize the calls of `submit_orders` of the agents in each step.
The number of workers is `simulation.numParallel`, which is the number of CPUs minus one (at least one) by default.
They give the same results as `SequentialRunner`, but they are not always faster.

- `MultiThreadAgentParallelRunner` does not speed up this simulation because of the GIL of Python: the Monte Carlo simulation is pure Python code, so only one thread runs it at a time. It is only slower by the overhead of the threads. This runner is beneficial only when agents wait for I/O, e.g., network access to external models or services.
- `MultiProcessAgentParallelRunner` runs the workloads on multiple processes, but it pickles the agent and the markets, which refer to the simulator, all the agents, and the logger, for every call of `submit_orders`. Only the orders and the state of the pseudo random number generator of the agent are sent back, so `bs_sum` stays 0.0 on the main process. The runner is faster than `SequentialRunner` only if the workload of each call is heavy enough to pay the cost of the pickling and multiple CPUs are available.

Here, a smaller simulation of 10 steps with 14 `WorkloadFCNAgent`s is run by each runner, changing the size of the workload, which is never shut down.
The results depend on the machine.

`MultiProcessAgentParallelRunner` is run only if the start method of multiprocessing is `fork`, e.g., on Linux (with Python 3.13 or older) including Google Colab.
With the other start methods, e.g., `spawn` on Windows and macOS, the classes defined in a notebook cannot be used by the worker processes.
In such a case, run the sample in the repository instead, whose classes are importable from the worker processes, e.g., `python -m samples.black_scholes.main --config samples/black_scholes/config.json --runner multi_process` from the root of the repository.
It is much slower than `--runner sequential` because the workload of the sample is small: it takes tens of seconds or more instead of less than a second, mostly for the pickling.

In [ ]:
benchmark_config = copy.deepcopy(config)
benchmark_config["simulation"]["sessions"][0]["iterationSteps"] = 10
benchmark_config["FCNAgent"]["numAgents"] = 4
benchmark_config["FCNAgents-I"]["numAgents"] = 2
benchmark_config["FCNAgent"]["bsLoadShutdown"] = -1
benchmark_config["FCNAgent"]["bsNumSteps"] = 100
n_samples_list = [1, 10, 100, 300]

runner_classes: List[Type[SequentialRunner]] = [SequentialRunner, MultiThreadAgentParallelRunner]
if multiprocessing.get_start_method() == "fork":
    runner_classes.append(MultiProcessAgentParallelRunner)
else:
    print(f"MultiProcessAgentParallelRunner is skipped because the start method of multiprocessing is {multiprocessing.get_start_method()}.")
runner_names = [runner_class.__name__ for runner_class in runner_classes]
print(f"number of CPUs: {os.cpu_count()}")

In [ ]:
results: Dict[Tuple[str, int], Tuple[MarketStepSaver, SequentialRunner, float]] = {}
for runner_class in runner_classes:
    for n_samples in n_samples_list:
        _config = copy.deepcopy(benchmark_config)
        _config["FCNAgent"]["bsNumSamples"] = n_samples
        results[(runner_class.__name__, n_samples)] = run(config=_config, runner_class=runner_class)

In [ ]:
# the results are identical to those of SequentialRunner
for runner_name in runner_names:
    for n_samples in n_samples_list:
        assert results[(runner_name, n_samples)][0].market_step_logs == results[("SequentialRunner", n_samples)][0].market_step_logs

n_mc_steps = [n_samples * benchmark_config["FCNAgent"]["bsNumSteps"] for n_samples in n_samples_list]
elapsed_times = {runner_name: np.array([results[(runner_name, n_samples)][2] for n_samples in n_samples_list]) for runner_name in runner_names}
speed_ups = {runner_name: elapsed_times["SequentialRunner"] / elapsed_times[runner_name] for runner_name in runner_names}
for runner_name in runner_names:
    _runner = results[(runner_name, n_samples_list[0])][1]
    if isinstance(_runner, MultiThreadAgentParallelRunner):
        print(f"{runner_name} ({_runner.num_parallel} workers)")
    else:
        print(runner_name)
    for n_mc_step, elapsed_time, speed_up in zip(n_mc_steps, elapsed_times[runner_name], speed_ups[runner_name]):
        print(f"  {n_mc_step} Monte Carlo steps per call: {elapsed_time:.2f} s (speed-up: {speed_up:.2f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for runner_name in runner_names:
    axes[0].plot(n_mc_steps, elapsed_times[runner_name], marker="o", label=runner_name)
    axes[1].plot(n_mc_steps, speed_ups[runner_name], marker="o", label=runner_name)
axes[0].set_yscale("log")
axes[0].set_ylabel("elapsed time [s]")
axes[1].axhline(1.0, color="black", linestyle="--")
axes[1].set_ylabel("speed-up from SequentialRunner")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("Monte Carlo steps per call (bsNumSamples x bsNumSteps)")
    ax.legend()
fig.tight_layout()
plt.show()

The speed-up of `MultiThreadAgentParallelRunner` stays around or below 1 for any size of the workload.
`MultiProcessAgentParallelRunner` is much slower than `SequentialRunner` for small workloads because of the pickling, and it can be faster only for heavy workloads on a machine with multiple CPUs.

The estimates are summed up to `bs_sum` on the worker processes, so they are lost with `MultiProcessAgentParallelRunner`.

In [ ]:
for runner_name in runner_names:
    _runner = results[(runner_name, n_samples_list[-1])][1]
    bs_sum = sum(agent.bs_sum for agent in _runner.simulator.agents if isinstance(agent, WorkloadFCNAgent))
    print(f"sum of bs_sum with {runner_name}: {bs_sum:.4f}")